# SheafPatternFusion Phase 2.5 - WP2.5.2 census patch - instance n4_r0199_d0

The six-shard audit covered 899/901 frame rows; this single n=4 structure (targets mean(0), mean(1)) failed in its shard. Runs the identical full-budget attack to complete the census to 319/319.

Code: pip-installed from hugogobato/sheafpatternfusion @ v0.3.0.
Runtime: CPU-only (~2 cores). Expected wall time: < 30 min.

First run: the first cell installs the pinned package and then HALTS with a message. Do Runtime > Restart session once (clears the preloaded numpy/scipy so the ABI-matched binaries load), then Runtime > Run all again; the install cell detects the pins and skips. Later runs on a warm session go straight through.

In [ ]:
import importlib.metadata as md
import subprocess
import sys

WANT = {'numpy': '2.4.3', 'scipy': '1.17.1', 'sheafpatternfusion': '0.3.0'}
TAG = 'v0.3.0'
REPO = 'https://github.com/hugogobato/sheafpatternfusion.git'


def _ver(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return None


missing = {p: v for p, v in WANT.items() if _ver(p) != v}
if not missing:
    print('environment OK:', WANT)
else:
    print('installing sheafpatternfusion@' + TAG + ' (one-time per session) ...')
    res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                          f'git+{REPO}@{TAG}'])
    if res.returncode != 0:
        raise RuntimeError('pip install failed; see log above')
    print()
    print('=' * 72)
    print('DEPENDENCIES INSTALLED. One manual step left:')
    print('  1) Runtime > Restart session ...   (clears the old numpy/scipy)')
    print('  2) Runtime > Run all               (this cell will skip)')
    print('=' * 72)
    raise SystemExit('restart required before importing numpy/scipy')


In [ ]:
import functools
import json
import multiprocessing as mp
import os
import pathlib
import time

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'


In [ ]:
import urllib.request
FROZEN_URL = 'https://raw.githubusercontent.com/hugogobato/sheafpatternfusion/v0.3.0/data/frozen/instances_merged.jsonl'
FROZEN_PATH = pathlib.Path('/content/instances_merged.jsonl')
if not FROZEN_PATH.exists():
    print('fetching frozen merge from', FROZEN_URL)
    urllib.request.urlretrieve(FROZEN_URL, FROZEN_PATH)
ROWS = [json.loads(l) for l in open(FROZEN_PATH)]
print('frozen merge:', len(ROWS), 'rows')


In [ ]:
BATTERY_CFG = json.loads(r'''{"description": "WP2.5.1 degeneracy null battery configuration. Frozen before any Phase-2.5 run.", "population": "engine-UNDETERMINED rows of data/frozen/instances_merged.jsonl", "tau_frac_observed": [0.3, 0.32, 0.34, 0.36, 0.38, 0.4, 0.42, 0.44, 0.46, 0.48, 0.5, 0.52, 0.54, 0.56, 0.58, 0.6, 0.62, 0.64, 0.66, 0.68, 0.7, 0.72, 0.74, 0.76, 0.78, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0], "tau_overlap": [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0], "tau_width": [0.001, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45], "priority_sample_cap": 200, "outputs": ["results/phase25/null_battery.json", "results/phase25/null_battery.csv", "results/phase25/priority_sample.jsonl"]}''')
AUDIT_CFG_FULL = json.loads(r'''{"description": "WP2.5.2 adversarial audit budgets. Phase 2's recorded engine costs (n=3 undecided rows: median 72 s, p90 303 s, max 677 s for round1+round2+fiber) make a literal x20 multiplier on every audited row infeasible inside the plan's own 30-60 CPU-hour envelope. Resolution per Section 11: A1 escalates ADAPTIVELY (rounds stop early on success or on no-improvement), so the x20 language is a per-row CEILING, not a flat spend; effective multiplier is data-dependent and is repriced by WP2.5.6 from the verdict logs. Every notebook self-pilots its first job and prints an extrapolation before continuing.", "sampling": {"n2_census": true, "n4_census": true, "n3_srs_N": 400, "srs_seed": 20250901, "include_S_star": true, "S_star_recomputed_in_notebook": true}, "full": {"a1_jump_rounds": 4, "a1_starts_per_round": 200, "a1_walk_n_seeds": 8, "a1_walk_steps": 60, "a1_step_size": 0.02, "a2_root_starts": 150, "a2_max_roots": 16, "a2_walk_follows": 4, "a2_walk_n_seeds": 8, "a2_walk_steps": 50, "a2_lp_vertices": 16, "a3_max_union_vars": 4, "a3_max_cells": 120, "a3_pin_tol": 1e-09, "phi_tol": 0.0001, "dist_tol": 1e-09}, "pilot": {"a1_jump_rounds": 1, "a1_starts_per_round": 20, "a1_walk_n_seeds": 2, "a1_walk_steps": 20, "a1_step_size": 0.02, "a2_root_starts": 24, "a2_max_roots": 8, "a2_walk_follows": 1, "a2_walk_n_seeds": 3, "a2_walk_steps": 20, "a2_lp_vertices": 6, "a3_max_union_vars": 4, "a3_max_cells": 40, "a3_pin_tol": 1e-09, "phi_tol": 0.0001, "dist_tol": 1e-09}, "adaptive_escalation": {"early_stop_on_success": true, "no_improvement_stop": "stop after a round whose best delta_phi improved <10% relative over the previous round, unless a witness was found", "note": "implemented in attackers.deepened_witness_search"}, "kill_rule": "any CONFIRMED false RECOVERABLE (model-valid pair: dist < dist_tol AND dphi > phi_tol on the attackers' independent FastFingerprint path) -> C1 KILL (rule D2)", "outputs": ["results/phase25/audit_sample.jsonl", "results/phase25/audit_verdicts.jsonl"], "a1_jump_rounds": 4, "a1_starts_per_round": 200, "a1_walk_n_seeds": 8, "a1_walk_steps": 60, "a1_step_size": 0.02, "a2_root_starts": 150, "a2_max_roots": 16, "a2_walk_follows": 4, "a2_walk_n_seeds": 8, "a2_walk_steps": 50, "a2_lp_vertices": 16, "a3_max_union_vars": 4, "a3_max_cells": 120, "a3_pin_tol": 1e-09, "phi_tol": 0.0001, "dist_tol": 1e-09}''')
AUDIT_CFG_PILOT = json.loads(r'''{"description": "WP2.5.2 adversarial audit budgets. Phase 2's recorded engine costs (n=3 undecided rows: median 72 s, p90 303 s, max 677 s for round1+round2+fiber) make a literal x20 multiplier on every audited row infeasible inside the plan's own 30-60 CPU-hour envelope. Resolution per Section 11: A1 escalates ADAPTIVELY (rounds stop early on success or on no-improvement), so the x20 language is a per-row CEILING, not a flat spend; effective multiplier is data-dependent and is repriced by WP2.5.6 from the verdict logs. Every notebook self-pilots its first job and prints an extrapolation before continuing.", "sampling": {"n2_census": true, "n4_census": true, "n3_srs_N": 400, "srs_seed": 20250901, "include_S_star": true, "S_star_recomputed_in_notebook": true}, "full": {"a1_jump_rounds": 4, "a1_starts_per_round": 200, "a1_walk_n_seeds": 8, "a1_walk_steps": 60, "a1_step_size": 0.02, "a2_root_starts": 150, "a2_max_roots": 16, "a2_walk_follows": 4, "a2_walk_n_seeds": 8, "a2_walk_steps": 50, "a2_lp_vertices": 16, "a3_max_union_vars": 4, "a3_max_cells": 120, "a3_pin_tol": 1e-09, "phi_tol": 0.0001, "dist_tol": 1e-09}, "pilot": {"a1_jump_rounds": 1, "a1_starts_per_round": 20, "a1_walk_n_seeds": 2, "a1_walk_steps": 20, "a1_step_size": 0.02, "a2_root_starts": 24, "a2_max_roots": 8, "a2_walk_follows": 1, "a2_walk_n_seeds": 3, "a2_walk_steps": 20, "a2_lp_vertices": 6, "a3_max_union_vars": 4, "a3_max_cells": 40, "a3_pin_tol": 1e-09, "phi_tol": 0.0001, "dist_tol": 1e-09}, "adaptive_escalation": {"early_stop_on_success": true, "no_improvement_stop": "stop after a round whose best delta_phi improved <10% relative over the previous round, unless a witness was found", "note": "implemented in attackers.deepened_witness_search"}, "kill_rule": "any CONFIRMED false RECOVERABLE (model-valid pair: dist < dist_tol AND dphi > phi_tol on the attackers' independent FastFingerprint path) -> C1 KILL (rule D2)", "outputs": ["results/phase25/audit_sample.jsonl", "results/phase25/audit_verdicts.jsonl"], "a1_jump_rounds": 1, "a1_starts_per_round": 20, "a1_walk_n_seeds": 2, "a1_walk_steps": 20, "a1_step_size": 0.02, "a2_root_starts": 24, "a2_max_roots": 8, "a2_walk_follows": 1, "a2_walk_n_seeds": 3, "a2_walk_steps": 20, "a2_lp_vertices": 6, "a3_max_union_vars": 4, "a3_max_cells": 40, "a3_pin_tol": 1e-09, "phi_tol": 0.0001, "dist_tol": 1e-09}''')


In [ ]:

OUT_DIR = pathlib.Path('/content/results/phase25')
OUT_DIR.mkdir(parents=True, exist_ok=True)


def load_done(path, key_fn):
    done = set()
    if path.exists():
        for line in path.read_text().splitlines():
            try:
                rec = json.loads(line)
                done.add(key_fn(rec))
            except Exception:
                pass
    return done


def pooled_map(worker_fn, items, n_workers=2,
               stall_timeout_s=2400):
    '''Yield worker_fn(item) for all items, 2-process pool with a stall
watchdog: if no future completes within stall_timeout_s, workers are killed
and the remainder runs sequentially. Worker_fn must be picklable (an
importable function or a functools.partial thereof).'''
    from concurrent.futures import ProcessPoolExecutor, FIRST_COMPLETED, wait

    items = list(items)
    if len(items) <= 1 or n_workers <= 1:
        for it in items:
            yield worker_fn(it)
        return
    ctx = mp.get_context('spawn')
    ex = ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx)
    done_count = 0
    try:
        futs = {ex.submit(worker_fn, it): it for it in items}
        pending = set(futs)
        while pending:
            done_set, pending = wait(pending, timeout=stall_timeout_s,
                                     return_when=FIRST_COMPLETED)
            if not done_set:
                raise RuntimeError(
                    f'pool stalled {stall_timeout_s}s with '
                    f'{len(pending)} futures pending')
            for f in done_set:
                done_count += 1
                yield f.result()
        ex.shutdown(wait=False, cancel_futures=True)
    except Exception as e:
        print(f'(pool yielded {done_count}/{len(items)} results, then '
              f'{type(e).__name__}; finishing remainder sequentially)',
              flush=True)
        for proc in (getattr(ex, '_processes', None) or {}).values():
            try:
                proc.kill()
            except Exception:
                pass
        ex.shutdown(wait=False, cancel_futures=True)
        for it in items[done_count:]:
            yield worker_fn(it)


In [ ]:
from sheafpatternfusion.workers import run_attack_job

JOBS = json.loads(r'''[{"instance_id": "n4_r0199_d0", "target": ["mean", 0], "seed": 25277070, "var_parents": {"0": [], "1": [], "2": [0, 1], "3": [0]}, "r_parents": [[2], [0, 1, 2, 3], [1], [0]]}, {"instance_id": "n4_r0199_d0", "target": ["mean", 1], "seed": 25277070, "var_parents": {"0": [], "1": [], "2": [0, 1], "3": [0]}, "r_parents": [[2], [0, 1, 2, 3], [1], [0]]}]''')
vpath = OUT_DIR / 'audit_verdicts_patch_n4r0199.jsonl'
done = load_done(vpath, lambda r: r['instance_id'] + '|' + json.dumps(r['target']))
pending = [j for j in JOBS if j['instance_id'] + '|' + json.dumps(j['target']) not in done]
print(f'{len(pending)} patch jobs to run (full budgets)')
for j in pending:
    t0 = time.time()
    rec = run_attack_job(dict(j), AUDIT_CFG_FULL)
    with open(vpath, 'a') as f:
        f.write(json.dumps(rec) + '\n')
    print(f"{rec['instance_id']} {rec['target']}: {rec['verdict']} "
          f"({time.time()-t0:.0f}s; A1 starts={rec['A1']['budget_starts']})", flush=True)
print('PATCH DONE')


In [ ]:
import glob
output_files = sorted(glob.glob(str(OUT_DIR / '*.jsonl')) + glob.glob(str(OUT_DIR / '*.json')))
for output_file in output_files:
    try:
        from google.colab import files
        files.download(output_file)
        print('Downloaded:', output_file)
    except Exception as e:
        print('(Not on Colab / download skipped):', e)
